In [5]:
import random
import math

In [6]:
class random_undirected_graph:
    def __init__(self, node, seed = None):
        self.node = node
        self.matrix = [[0]*node for _ in range(node)]
        self.seed = seed
        self.generate_distance()
    def generate_distance(self):
        if self.seed is not None:
            random.seed(self.seed)
        # generate random points
        points = [(random.uniform(0,100),random.uniform(0,100)) for _ in range(self.node)] ## (1,2)

        # calculate distance
        for i in range(self.node):
            for j in range(self.node):
                if i == j:
                    self.matrix[i][j] = 0
                else:
                    dx = points[i][0] - points[j][0]
                    dy = points[i][1] - points[j][1]
                    dist = int(math.sqrt(dx**2 + dy**2))

                    self.matrix[i][j] = dist
                    self.matrix[j][i] = dist  # enforce symmetry
    def print_matrix(self):
        for row in self.matrix:
            print(" ".join(f"{val:6.2f}" for val in row))

In [8]:
class prims_MST:
    def __init__(self, random_undirected_graph, start_node=0):
        self.start_node = start_node
        self.graph = random_undirected_graph.matrix;
        self.n = len(self.graph)
        self.parent = [-1]*self.n
        self.dist = [float('inf')]*self.n
        self.visited = [False]*self.n
    def compute_mst(self):
        self.dist[self.start_node] = 0

        for _ in range(self.n):
            u = self._min_distance() # calculate the minimum distance of node
            self.visited[u] = True
            for v in range(self.n):
                weight = self.graph[u][v]
                if weight > 0 and not self.visited[v] and weight < self.dist[v]:
                    self.parent[v] = u
                    self.dist[v] = weight

    def _min_distance(self):
        min_dist = float('inf')
        min_index = -1

        for v in range(self.n):
            if not self.visited[v] and self.dist[v] < min_dist:
                min_dist = self.dist[v]
                min_index = v

        return min_index


In [15]:
class approx_TSP:
    def __init__(self, graph_obj, starting_tour=0):
        self.graph_obj = graph_obj
        self.graph = graph_obj.matrix
        self.starting_tour = starting_tour
        self.n = len(self.graph)
        self.tour = []
        self.mst_adj = [[] for _ in range(self.n)]
        self.total_cost = 0

    def compute_tsp(self):
        mst = prims_MST(self.graph_obj, self.starting_tour)
        mst.compute_mst()

        # Build MST adjacency list (skip root)
        for u in range(self.n):
            v = mst.parent[u]
            if v != -1:
                self.mst_adj[u].append(v)
                self.mst_adj[v].append(u)

        visited = [False]*self.n
        self._dfs(self.starting_tour, visited)
        self.tour.append(self.tour[0])

        # compute tour cost
        for i in range(len(self.tour)-1):
            self.total_cost += self.graph[self.tour[i]][self.tour[i+1]]

    def _dfs(self, u, visited):
        visited[u] = True
        self.tour.append(u)
        for v in self.mst_adj[u]:
            if not visited[v]:
                self._dfs(v, visited)

    def Print_Tour(self):
        print("TSP Tour:")
        print(" -> ".join(map(str, self.tour)))
        print("Total Tour Cost:", self.total_cost)

In [16]:
initial_graph = random_undirected_graph(node=5,seed=100)
initial_graph.print_matrix()
tsp = approx_TSP(initial_graph,starting_tour=0)
tsp.compute_tsp()
print("\n2-Approximate TSP Solution:")
tsp.Print_Tour()

  0.00  67.00  58.00  65.00   6.00
 67.00   0.00  27.00  17.00  73.00
 58.00  27.00   0.00  12.00  65.00
 65.00  17.00  12.00   0.00  72.00
  6.00  73.00  65.00  72.00   0.00

2-Approximate TSP Solution:
TSP Tour:
0 -> 2 -> 3 -> 1 -> 4 -> 0
Total Tour Cost: 166
